In [2]:
!pip install qiskit
!pip install qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 114.8 MB/s eta 0:00:00


In [16]:
!pip install -q qiskit_ibm_runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.6/378.6 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 11.1 MB/s eta 0:00:00


In [35]:
from qiskit import QuantumCircuit
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Kraus, SuperOp
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

# Import from Qiskit Aer noise module
from qiskit_aer.noise import (
    NoiseModel,
    QuantumError,
    ReadoutError,
    depolarizing_error,
    pauli_error,
    thermal_relaxation_error,
)
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import SamplerV2 as Sampler
backend = AerSimulator()
sampler = Sampler(mode=backend)

In [36]:
def prepare_bb84_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)

    if bit == 1:
        qc.x(0)
    if basis == 1:  # X basis
        qc.h(0)

    return qc


In [87]:
def bb84_single_shot(alice_bit, alice_basis, bob_basis):
    qc = prepare_bb84_qubit(alice_bit, alice_basis)

    if bob_basis == 1:
        qc.h(0)

    qc.measure(0, 0)

    job = noisy_sampler.run([qc], shots=1)
    result = job.result()
    counts = result[0].data.c.get_counts(0)
    key = next(iter(counts))
    measured_bit=int(key)
    return measured_bit


In [38]:
def bb84_baseline_qiskit(N=200):
    alice_bits = np.random.randint(0, 2, N)
    alice_bases = np.random.randint(0, 2, N)
    bob_bases = np.random.randint(0, 2, N)

    bob_results = []

    for i in range(N):
        bob_results.append(
            bb84_single_shot(
                alice_bits[i],
                alice_bases[i],
                bob_bases[i]
            )
        )

    # Sifting
    mask = alice_bases == bob_bases
    alice_key = alice_bits[mask]
    bob_key = np.array(bob_results)[mask]

    qber = np.mean(alice_key != bob_key)
    return len(alice_key), qber


In [39]:
def intercept_resend_qiskit(N=200):
    alice_bits = np.random.randint(0, 2, N)
    alice_bases = np.random.randint(0, 2, N)
    eve_bases = np.random.randint(0, 2, N)
    bob_bases = np.random.randint(0, 2, N)

    bob_results = []
    eve_results = []

    for i in range(N):
        # Alice → Eve
        eve_bit = bb84_single_shot(
            alice_bits[i],
            alice_bases[i],
            eve_bases[i]
        )
        eve_results.append(eve_bit)

        # Eve → Bob
        bob_bit = bb84_single_shot(
            eve_bit,
            eve_bases[i],
            bob_bases[i]
        )
        bob_results.append(bob_bit)

    mask = alice_bases == bob_bases
    alice_key = alice_bits[mask]
    bob_key = np.array(bob_results)[mask]
    eve_key = np.array(eve_results)[mask]

    qber = np.mean(alice_key != bob_key)
    eve_info = np.mean(alice_key == eve_key)

    return len(alice_key), qber, eve_info


In [40]:
noise_model = NoiseModel.from_backend(backend)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.02, 1),
    ['x', 'h']
)


/usr/local/lib/python3.12/dist-packages/qiskit_aer/noise/noise_model.py:376: UserWarning: Qiskit backend AerSimulator('aer_simulator') has no QubitProperties, so the resulting noise model will not include any thermal relaxation errors.
  warn(


In [88]:
def print_result(title, key_len, qber, eve_info=None):
    print(f"\n{title}")
    print(f"  Sifted Key Length : {key_len}")
    print(f"  QBER              : {qber:.5f}")
    if eve_info is not None:
        print(f"  Eve Info          : {eve_info:.5f}")
print("===== QISKIT BB84 — IDEAL CHANNEL =====")

key_len, qber = bb84_baseline_qiskit(N=200)
print_result("Baseline (No Eve)", key_len, qber)
key_len, qber, eve_info = intercept_resend_qiskit(N=200)
print_result("Intercept–Resend Attack", key_len, qber, eve_info)

def bb84_single_shot_noisy(alice_bit, alice_basis, bob_basis, noise_model):
    qc = prepare_bb84_qubit(alice_bit, alice_basis)

    if bob_basis == 1:
        qc.h(0)

    qc.measure(0, 0)

    job = execute(
        qc,
        backend,
        shots=1,
        noise_model=noise_model
    )
    result = job.result().get_counts()
    return int(list(result.keys())[0])
def bb84_baseline_qiskit_noisy(N=200, noise_model=None):
    alice_bits = np.random.randint(0, 2, N)
    alice_bases = np.random.randint(0, 2, N)
    bob_bases = np.random.randint(0, 2, N)

    bob_results = []

    for i in range(N):
        bob_results.append(
            bb84_single_shot_noisy(
                alice_bits[i],
                alice_bases[i],
                bob_bases[i],
                noise_model
            )
        )

    mask = alice_bases == bob_bases
    alice_key = alice_bits[mask]
    bob_key = np.array(bob_results)[mask]

    qber = np.mean(alice_key != bob_key)
    return len(alice_key), qber
def intercept_resend_qiskit_noisy(N=200, noise_model=None):
    alice_bits = np.random.randint(0, 2, N)
    alice_bases = np.random.randint(0, 2, N)
    eve_bases = np.random.randint(0, 2, N)
    bob_bases = np.random.randint(0, 2, N)

    bob_results = []
    eve_results = []

    for i in range(N):
        # Alice → Eve
        eve_bit = bb84_single_shot_noisy(
            alice_bits[i],
            alice_bases[i],
            eve_bases[i],
            noise_model
        )
        eve_results.append(eve_bit)

        # Eve → Bob
        bob_bit = bb84_single_shot_noisy(
            eve_bit,
            eve_bases[i],
            bob_bases[i],
            noise_model
        )
        bob_results.append(bob_bit)

    mask = alice_bases == bob_bases
    alice_key = alice_bits[mask]
    bob_key = np.array(bob_results)[mask]
    eve_key = np.array(eve_results)[mask]

    qber = np.mean(alice_key != bob_key)
    eve_info = np.mean(alice_key == eve_key)

    return len(alice_key), qber, eve_info


===== QISKIT BB84 — IDEAL CHANNEL =====

Baseline (No Eve)
  Sifted Key Length : 96
  QBER              : 0.01042

Intercept–Resend Attack
  Sifted Key Length : 109
  QBER              : 0.25688
  Eve Info          : 0.78899


In [56]:
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler

noise_model = NoiseModel()
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(0.02, 1),
    ['x', 'h']
)

noisy_backend = AerSimulator(noise_model=noise_model)
noisy_sampler = Sampler(mode=noisy_backend)


In [85]:
def bb84_single_shot_noisy(alice_bit, alice_basis, bob_basis):
    qc = prepare_bb84_qubit(alice_bit, alice_basis)

    if bob_basis == 1:
        qc.h(0)

    qc.measure(0, 0)

    job = noisy_sampler.run([qc], shots=1)
    result = job.result()
    counts = result[0].data.c.get_counts(0)
    key = next(iter(counts))
    measured_bit=int(key)
    return measured_bit


In [86]:
noisy_bit = bb84_single_shot_noisy(
    alice_bit=1,
    alice_basis=0,
    bob_basis=1
)
print("Noisy:", noisy_bit)

Noisy: 1


In [49]:
print("\n===== QISKIT BB84 — NOISY CHANNEL =====")

key_len, qber = bb84_baseline_qiskit_noisy(
    N=200,
    noise_model=noise_model
)
print_result("Baseline (No Eve)", key_len, qber)

key_len, qber, eve_info = intercept_resend_qiskit_noisy(
    N=200,
    noise_model=noise_model
)
print_result("Intercept–Resend Attack", key_len, qber, eve_info)




===== QISKIT BB84 — NOISY CHANNEL =====


NameError: name 'execute' is not defined